In [ ]:

import numpy as np
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score


# ==========================================
# STEP 1: GENERATE DATA
# ==========================================

def make_data(seed=42):

    X, y = make_blobs(
        n_samples=300,
        centers=4,
        n_features=2,
        cluster_std=1.0,
        random_state=seed
    )

    return X, y


# ==========================================
# STEP 2: EUCLIDEAN DISTANCE
# ==========================================

def euclidean(p, q):

    p = np.asarray(p)
    q = np.asarray(q)

    return float(np.sqrt(np.sum((p - q) ** 2)))


# ==========================================
# STEP 3: ASSIGN CLUSTERS
# ==========================================

def assign_clusters(X, centroids):

    labels = []

    for point in X:

        distances = []

        for centroid in centroids:

            distances.append(euclidean(point, centroid))

        labels.append(np.argmin(distances))

    return np.array(labels, dtype=int)


# ==========================================
# STEP 4: UPDATE CENTROIDS
# ==========================================

def update_centroids(X, labels, k):

    X = np.asarray(X)
    labels = np.asarray(labels)

    n_features = X.shape[1]

    centroids = np.zeros((k, n_features))

    for j in range(k):

        cluster_points = X[labels == j]

        if len(cluster_points) > 0:
            centroids[j] = np.mean(cluster_points, axis=0)

    return centroids


# ==========================================
# STEP 5: COMPUTE INERTIA
# ==========================================

def compute_inertia(X, labels, centroids):

    X = np.asarray(X)
    labels = np.asarray(labels)

    inertia = 0.0

    for i in range(len(X)):

        centroid = centroids[labels[i]]

        distance = euclidean(X[i], centroid)

        inertia += distance ** 2

    return float(inertia)


# ==========================================
# STEP 6: K-MEANS ALGORITHM
# ==========================================

def kmeans(
    X,
    k,
    init_centroids,
    max_iter=100,
    tol=1e-6
):

    X = np.asarray(X, dtype=float)

    centroids = np.asarray(
        init_centroids,
        dtype=float
    ).copy()

    for iteration in range(max_iter):

        # Assign each point to nearest centroid
        labels = assign_clusters(X, centroids)

        # Calculate new centroids
        new_centroids = update_centroids(
            X,
            labels,
            k
        )

        # Calculate centroid movement
        movement = np.sqrt(
            np.sum((new_centroids - centroids) ** 2)
        )

        # Update centroids
        centroids = new_centroids

        # Stop if movement is within tolerance
        if movement <= tol:
            break

    # Final assignment using updated centroids
    labels = assign_clusters(X, centroids)

    # Calculate final inertia
    inertia = compute_inertia(
        X,
        labels,
        centroids
    )

    return labels, centroids, inertia


# ==========================================
# STEP 7: VERIFY EUCLIDEAN DISTANCE
# ==========================================

print("Euclidean Distance:")
print(euclidean([0, 0], [3, 4]))

# Expected: 5.0


# ==========================================
# STEP 8: VERIFY CLUSTER ASSIGNMENT
# ==========================================

Xs = np.array([
    [0, 0],
    [0, 1],
    [10, 10],
    [10, 11]
], dtype=float)

Cs = np.array([
    [0, 0.5],
    [10, 10.5]
], dtype=float)

print("\nCluster Assignments:")
print(assign_clusters(Xs, Cs).tolist())

# Expected: [0, 0, 1, 1]


# ==========================================
# STEP 9: VERIFY CENTROID UPDATE
# ==========================================

test_labels = np.array([0, 0, 1, 1])

updated = update_centroids(
    Xs,
    test_labels,
    2
)

print("\nUpdated Centroids:")
print(updated.tolist())

# Expected:
# [[0.0, 0.5], [10.0, 10.5]]


# ==========================================
# STEP 10: VERIFY INERTIA
# ==========================================

inertia_test = compute_inertia(
    Xs,
    test_labels,
    Cs
)

print("\nTest Inertia:")
print(round(inertia_test, 3))

# Expected: 1.0


# ==========================================
# STEP 11: GENERATE ACTUAL DATASET
# ==========================================

X, y = make_data(42)

print("\nDataset Shape:")
print("X:", X.shape)
print("y:", y.shape)


# ==========================================
# STEP 12: INITIALIZE CENTROIDS
# ==========================================

init = X[[0, 75, 150, 225]].astype(float)

print("\nInitial Centroids:")
print(init)


# ==========================================
# STEP 13: RUN K-MEANS
# ==========================================

labels, centroids, inertia = kmeans(
    X,
    4,
    init
)

print("\nFinal Centroids:")
print(centroids)

print("\nFinal Inertia:")
print(round(inertia, 1))

print("\nCluster Labels:")
print(labels)


# ==========================================
# STEP 14: VERIFY CLUSTER AGREEMENT
# ==========================================

ari = adjusted_rand_score(y, labels)

print("\nAdjusted Rand Index:")
print(round(ari, 3))


# ==========================================
# STEP 15: VERIFY INITIAL VS FINAL INERTIA
# ==========================================

initial_labels = assign_clusters(X, init)

initial_inertia = compute_inertia(
    X,
    initial_labels,
    init
)

print("\nInitial Inertia:")
print(round(initial_inertia, 3))

print("\nFinal Inertia:")
print(round(inertia, 3))

print(
    "\nInertia Decreased:",
    inertia <= initial_inertia
)
